# Reconstructing the AMOC through EOF dimentionality reduction

## EOF analysis

### Subsurface longitudal fields (latitude-depth)

In [ ]:
# ============================================================
# Combined EOF Analysis for CMIP6 ocean variables in (lat,depth)
# at fixed longitudes
#
# EOFs fit on TRAIN only; PCs are projected for FULL period.
# ============================================================

import os
import glob
import re
import numpy as np
import xarray as xr
from eofs.standard import Eof
import warnings
warnings.filterwarnings("ignore")

# Prevent HDF5 file locking issues on shared filesystems
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# CONFIG
# ============================================================
MODEL   = "IPSL-CM6A-LR"      # EC-Earth3, IPSL-CM6A-LR, CESM2, MPI-ESM1-2-LR
VAR     = "so"             # thetao or so
NMODES  = 10

START_YEAR = 1850
END_YEAR   = 2014

TARGET_LONS   = [-10.0, -20.0, -30.0, -40.0, -60.0]
LON_WINDOW    = 10.0       # degrees
DEPTH_RANGE_M = (0.0, 5000.0)

# >>> NEW: Train fraction of the COMMON time axis (first X%)
TRAIN_FRACTION = 0.85  # choose 0.80–0.85 as you like

# ============================================================
# DIRECTORIES
# ============================================================
if MODEL == "EC-Earth3":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/EC-Earth3/{VAR}/masked/"
elif MODEL == "IPSL-CM6A-LR":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/IPSL-CM6A-LR/{VAR}/masked/"
elif MODEL == "CESM2":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/CESM2/{VAR}/masked/"
elif MODEL == "MPI-ESM1-2-LR":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/MPI-ESM1-2-LR/{VAR}/masked/"
else:
    raise ValueError("Unknown MODEL")

OUT_DIR = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/latdepth_sections/Train_period_{int(TRAIN_FRACTION*100)}pct/"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# UTILITIES
# ============================================================

def parse_member_label(path):
    m = re.search(r"_(r\d+i\d+p\d+f\d+)", os.path.basename(path))
    return m.group(1) if m else os.path.basename(path)

def wrap_lon(lon):
    return ((lon + 180) % 360) - 180

def standardize_latlon(ds):
    lat_var = next(v for v in ["lat", "latitude", "nav_lat"] if v in ds)
    lon_var = next(v for v in ["lon", "longitude", "nav_lon"] if v in ds)

    lat = ds[lat_var]
    lon = ds[lon_var]

    if lat.ndim == 1 and lon.ndim == 1:
        lon2d, lat2d = np.meshgrid(lon.values, lat.values)
        ds = ds.drop_vars([lat_var, lon_var])
        ds = ds.rename_dims({lat.dims[0]: "y", lon.dims[0]: "x"})
        ds["lat2d"] = xr.DataArray(lat2d, dims=("y", "x"))
        ds["lon2d"] = xr.DataArray(lon2d, dims=("y", "x"))
        return ds

    ds = ds.swap_dims({lat.dims[0]: "y", lat.dims[1]: "x"})
    ds = ds.rename({lat_var: "lat2d", lon_var: "lon2d"})
    return ds

def get_depth_dim_and_coord(da, ds):
    if "olevel" in da.dims:
        return "olevel", ds["olevel"].values
    if "depth" in da.dims:
        return "depth", ds["depth"].values
    if "lev" in da.dims:
        z = ds["lev"].values
        if np.nanmax(z) > 10000:
            z = z / 100.0
        return "lev", z
    raise KeyError("No vertical coordinate found")

def compute_layer_thickness(z):
    z = np.asarray(z)
    dz = np.empty_like(z, dtype=float)
    dz[1:-1] = 0.5 * (z[2:] - z[:-2])
    dz[0]    = z[1] - z[0]
    dz[-1]   = z[-1] - z[-2]
    return np.abs(dz)

def extract_section_valid_ocean(da, ds, target_lon, depth_dim):
    lon2d = ds["lon2d"].values
    lat2d = ds["lat2d"].values

    lonW = wrap_lon(lon2d)
    tW   = wrap_lon(target_lon)

    valid_xy = np.isfinite(da).any(dim=("time", depth_dim)).values
    ny, nx = lonW.shape

    ix = np.full(ny, -1, dtype=int)

    for y in range(ny):
        cand = np.where(valid_xy[y])[0]
        if cand.size == 0:
            continue

        d = np.abs(lonW[y, cand] - tW)
        ok = d <= LON_WINDOW
        if not np.any(ok):
            continue

        cand = cand[ok]
        d    = d[ok]
        ix[y] = cand[np.argmin(d)]

    valid_row = ix >= 0

    sec = da.isel(x=xr.DataArray(np.where(valid_row, ix, 0), dims="y"))
    sec = sec.where(xr.DataArray(valid_row, dims="y"))

    lat_1d = np.where(valid_row, lat2d[np.arange(ny), ix], np.nan)

    return sec, lat_1d, valid_row

# ============================================================
# MAIN WORKFLOW
# ============================================================

def run_longitude(target_lon):

    print("\n" + "="*80)
    print(f"▶ Longitude {target_lon:.1f}° | MODEL={MODEL} | VAR={VAR}")
    print("="*80)

    files = sorted(glob.glob(os.path.join(IN_DIR, f"*{VAR}*_masked*.nc")))
    if not files:
        raise FileNotFoundError("No masked files found")

    sections = []
    member_labels = []

    lat_ref = z_ref = dz_ref = depth_dim_ref = None

    for i, f in enumerate(files, start=1):
        label = parse_member_label(f)
        member_labels.append(label)

        print(f"\n  → Member {i}/{len(files)}: {label}")

        ds = standardize_latlon(xr.open_dataset(f))
        da = ds[VAR]

        depth_dim, z = get_depth_dim_and_coord(da, ds)
        dz = compute_layer_thickness(z)

        da = da.sel(time=slice(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"))

        sec, lat1d, valid = extract_section_valid_ocean(
            da, ds, target_lon, depth_dim
        )

        keepz = (z >= DEPTH_RANGE_M[0]) & (z <= DEPTH_RANGE_M[1])
        sec = sec.isel({depth_dim: keepz})
        z_sel  = z[keepz]
        dz_sel = dz[keepz]

        good = np.isfinite(lat1d)
        sec = sec.isel(y=good)
        lat1d = lat1d[good]

        order = np.argsort(lat1d)
        sec = sec.isel(y=order)
        lat1d = lat1d[order]

        # anomaly over the full period; TRAIN solver will still be fit only on TRAIN time
        sec = sec - sec.mean("time")
        sec = sec.expand_dims(member=[label])
        sections.append(sec)

        if lat_ref is None:
            lat_ref = lat1d
            z_ref   = z_sel
            dz_ref  = dz_sel
            depth_dim_ref = depth_dim

        print(f"    Kept latitude rows: {lat1d.size}")

    print("\n▶ Aligning time across members")
    common_time = sections[0]["time"].values
    for s in sections[1:]:
        common_time = np.intersect1d(common_time, s["time"].values)

    # >>> NEW: ensure chronological order
    common_time = np.sort(common_time)

    combined = xr.concat([s.sel(time=common_time) for s in sections], dim="member")

    # SAFE renaming: avoid 'lat' name collision
    combined = combined.rename({
        "y": "lat_index",
        depth_dim_ref: "depth"
    })

    combined = combined.assign_coords(
        lat=("lat_index", lat_ref),
        depth=("depth", z_ref),
    )

    n_member = combined.sizes["member"]
    n_time   = combined.sizes["time"]
    n_lat    = combined.sizes["lat_index"]
    n_depth  = combined.sizes["depth"]

    # >>> NEW: define TRAIN time split on the common axis
    n_train = int(np.floor(TRAIN_FRACTION * n_time))
    n_train = max(2, min(n_train, n_time))  # safety
    train_time = combined["time"].values[:n_train]

    train_mask = np.zeros(n_time, dtype=bool)
    train_mask[:n_train] = True

    print(f"\n▶ Combined shape: member={n_member}, time={n_time}, lat={n_lat}, depth={n_depth}")
    print(f"▶ TRAIN_FRACTION={TRAIN_FRACTION} -> n_train={n_train}/{n_time} "
          f"(train end = {str(train_time[-1])})")

    # FULL data (for projection)
    data_full = combined.transpose("member", "time", "lat_index", "depth").values
    data2d_full = data_full.reshape(n_member * n_time, n_lat * n_depth)
    data2d_full = np.ma.masked_invalid(data2d_full)

    # TRAIN data (for fitting EOFs)
    combined_train = combined.sel(time=train_time)
    data_train = combined_train.transpose("member", "time", "lat_index", "depth").values
    data2d_train = data_train.reshape(n_member * n_train, n_lat * n_depth)
    data2d_train = np.ma.masked_invalid(data2d_train)

    print("▶ Computing EOF weights")
    w_lat = np.clip(np.cos(np.deg2rad(lat_ref)), 0, None)
    w2d   = np.sqrt(w_lat[:, None] * dz_ref[None, :])
    weights = w2d.reshape(n_lat * n_depth)

    print("▶ Solving EOFs on TRAIN only")
    solver = Eof(data2d_train, weights=weights)

    EOFs = solver.eofs(neofs=NMODES).reshape(NMODES, n_lat, n_depth)
    VF   = solver.varianceFraction()[:NMODES]

    # >>> NEW: project FULL period onto TRAIN EOFs
    # projectField returns (n_samples, neofs) where n_samples = n_member*n_time
    PCs_full_flat = solver.projectField(data2d_full, neofs=NMODES)
    PCs = np.asarray(PCs_full_flat).reshape(n_member, n_time, NMODES)

    tag = f"{abs(target_lon):04.1f}".replace(".", "p")
    hemi = "W" if target_lon < 0 else "E"
    outfile = os.path.join(OUT_DIR, f"EOF_latdepth_{VAR}_{hemi}{tag}.nc")

    print(f"▶ Writing {outfile}")

    xr.Dataset(
        {
            "EOF": (("mode", "lat_index", "depth"), EOFs),
            "PC":  (("member", "time", "mode"), PCs),
            "variance_fraction": (("mode",), VF),
            "train_mask": (("time",), train_mask),
        },
        coords={
            "mode": np.arange(1, NMODES + 1),
            "member": np.array(member_labels, dtype=object),
            "time": combined["time"].values,
            "lat": ("lat_index", lat_ref),
            "depth": ("depth", z_ref),
        },
        attrs={
            "MODEL": MODEL,
            "VAR": VAR,
            "TARGET_LON_DEG": target_lon,
            "LON_WINDOW_DEG": LON_WINDOW,
            "WEIGHTING": "sqrt(cos(lat) * dz)",
            "DEPTH_RANGE_M": f"{DEPTH_RANGE_M[0]}-{DEPTH_RANGE_M[1]}",
            "EOF_FIT": "TRAIN_ONLY",
            "TRAIN_FRACTION": float(TRAIN_FRACTION),
            "TRAIN_START": str(train_time[0]),
            "TRAIN_END": str(train_time[-1]),
        }
    ).to_netcdf(outfile)

    print(f"✓ Done longitude {target_lon:.1f}°")

# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    print(f"\n=== MODEL={MODEL} | VAR={VAR} ===")
    print(f"TARGET_LONS={TARGET_LONS}")
    print(f"LON_WINDOW={LON_WINDOW} deg")
    print(f"DEPTH_RANGE_M={DEPTH_RANGE_M}")
    print(f"TRAIN_FRACTION={TRAIN_FRACTION}")

    for lon in TARGET_LONS:
        run_longitude(lon)

    print("\n✅ All longitudes processed successfully.")


### Surface fields (latitude-longitude)

In [ ]:
# ============================================================
# Combined SURFACE EOF Analysis for CMIP6 ocean variables in (lat,lon)
# over the masked Atlantic domain
#
# EOFs fit on TRAIN only; PCs are projected for FULL period.
# ============================================================

import os
import glob
import re
import numpy as np
import xarray as xr
from eofs.standard import Eof
import warnings
warnings.filterwarnings("ignore")

# Prevent HDF5 file locking issues on shared filesystems
os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# CONFIG
# ============================================================
MODEL   = "EC-Earth3"   # EC-Earth3, IPSL-CM6A-LR, CESM2, MPI-ESM1-2-LR
VAR     = "so"            # thetao or so
NMODES  = 10

START_YEAR = 1850
END_YEAR   = 2014

# TRAIN split on the COMMON time axis
TRAIN_FRACTION = 0.85

# If True: subtract temporal mean at each grid point over FULL period
# before TRAIN-only EOF fitting (same philosophy as your current script)
REMOVE_TEMPORAL_MEAN = True

# ============================================================
# DIRECTORIES
# ============================================================
if MODEL == "EC-Earth3":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/EC-Earth3/{VAR}/masked/"
elif MODEL == "IPSL-CM6A-LR":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/IPSL-CM6A-LR/{VAR}/masked/"
elif MODEL == "CESM2":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/CESM2/{VAR}/masked/"
elif MODEL == "MPI-ESM1-2-LR":
    IN_DIR = f"/data/projects/nckf/frekle/CMIP6_data/MPI-ESM1-2-LR/{VAR}/masked/"
else:
    raise ValueError("Unknown MODEL")

OUT_DIR = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/surface_latlon/Train_period_{int(TRAIN_FRACTION*100)}pct/"
os.makedirs(OUT_DIR, exist_ok=True)

# ============================================================
# UTILITIES
# ============================================================

def parse_member_label(path):
    m = re.search(r"_(r\d+i\d+p\d+f\d+)", os.path.basename(path))
    return m.group(1) if m else os.path.basename(path)

def wrap_lon(lon):
    return ((lon + 180) % 360) - 180

def standardize_latlon(ds):
    """
    Convert model-specific horizontal coordinates to:
      - dims: y, x
      - coords/variables: lat2d(y,x), lon2d(y,x)

    Works for both rectilinear and curvilinear grids.
    """
    lat_var = next(v for v in ["lat", "latitude", "nav_lat"] if v in ds)
    lon_var = next(v for v in ["lon", "longitude", "nav_lon"] if v in ds)

    lat = ds[lat_var]
    lon = ds[lon_var]

    # --------------------------------------------------------
    # Case 1: rectilinear grid -> lat(y), lon(x)
    # --------------------------------------------------------
    if lat.ndim == 1 and lon.ndim == 1:
        ydim = lat.dims[0]
        xdim = lon.dims[0]

        # rename dims only if needed
        dim_map = {}
        if ydim != "y":
            dim_map[ydim] = "y"
        if xdim != "x":
            dim_map[xdim] = "x"
        if dim_map:
            ds = ds.rename_dims(dim_map)

        # refresh objects after possible dim rename
        lat = ds[lat_var]
        lon = ds[lon_var]

        lon2d, lat2d = np.meshgrid(lon.values, lat.values)

        # rename original coordinate variables only if needed
        var_map = {}
        if lat_var != "lat_old":
            var_map[lat_var] = "lat_old"
        if lon_var != "lon_old":
            var_map[lon_var] = "lon_old"
        ds = ds.rename(var_map)

        ds["lat2d"] = xr.DataArray(lat2d, dims=("y", "x"))
        ds["lon2d"] = xr.DataArray(lon2d, dims=("y", "x"))
        return ds

    # --------------------------------------------------------
    # Case 2: curvilinear grid -> lat(y,x), lon(y,x)
    # --------------------------------------------------------
    elif lat.ndim == 2 and lon.ndim == 2:
        ydim, xdim = lat.dims

        # rename dims only if needed
        dim_map = {}
        if ydim != "y":
            dim_map[ydim] = "y"
        if xdim != "x":
            dim_map[xdim] = "x"
        if dim_map:
            ds = ds.rename_dims(dim_map)

        # rename lat/lon variables only if needed
        var_map = {}
        if lat_var != "lat2d":
            var_map[lat_var] = "lat2d"
        if lon_var != "lon2d":
            var_map[lon_var] = "lon2d"
        if var_map:
            ds = ds.rename(var_map)

        return ds

    else:
        raise ValueError(
            f"Unsupported lat/lon coordinate structure: "
            f"lat.ndim={lat.ndim}, lon.ndim={lon.ndim}, "
            f"lat.dims={lat.dims}, lon.dims={lon.dims}"
        )

def get_depth_dim_and_coord(da, ds):
    """
    Return vertical dimension name and depth values in meters.
    If no vertical dimension exists, return (None, None).
    """
    if "olevel" in da.dims:
        return "olevel", np.asarray(ds["olevel"].values, dtype=float)
    if "depth" in da.dims:
        return "depth", np.asarray(ds["depth"].values, dtype=float)
    if "lev" in da.dims:
        z = np.asarray(ds["lev"].values, dtype=float)
        # CESM often stores lev in centimeters
        if np.nanmax(z) > 10000:
            z = z / 100.0
        return "lev", z
    return None, None

def select_surface_field(da, ds):
    """
    Select the shallowest model level if a vertical dimension exists.
    Returns:
      da_surf  : (time, y, x)
      surf_z_m : scalar depth in meters, or NaN if no z dimension exists
      z_name   : vertical dim name or None
    """
    z_name, z = get_depth_dim_and_coord(da, ds)

    if z_name is None:
        # already 2D horizontal field
        return da, np.nan, None

    iz = int(np.nanargmin(z))
    surf_z_m = float(z[iz])

    da_surf = da.isel({z_name: iz})
    return da_surf, surf_z_m, z_name

def make_space_mask_from_train(data_train):
    """
    data_train shape: (member, time, y, x)
    Returns a boolean mask of spatial points valid for ALL train samples.
    This guarantees a constant mask for EOF solving.
    """
    valid_space = np.all(np.isfinite(data_train), axis=(0, 1))
    return valid_space

def flatten_valid_space(data4d, valid_space):
    """
    data4d shape: (member, time, y, x)
    valid_space shape: (y, x)
    Returns 2D array: (member*time, n_valid_space)
    """
    n_member, n_time, ny, nx = data4d.shape
    flat = data4d.reshape(n_member * n_time, ny * nx)
    keep = valid_space.reshape(ny * nx)
    return flat[:, keep]

def unflatten_eofs_to_fullgrid(eofs_valid, valid_space, ny, nx):
    """
    eofs_valid shape: (mode, n_valid_space)
    Returns full-grid EOFs: (mode, ny, nx) with NaN over invalid cells.
    """
    nmode = eofs_valid.shape[0]
    out = np.full((nmode, ny * nx), np.nan, dtype=float)
    out[:, valid_space.reshape(ny * nx)] = eofs_valid
    return out.reshape(nmode, ny, nx)

# ============================================================
# MAIN WORKFLOW
# ============================================================

def run_surface_eof():

    print("\n" + "="*80)
    print(f"▶ SURFACE LAT-LON EOF | MODEL={MODEL} | VAR={VAR}")
    print("="*80)

    files = sorted(glob.glob(os.path.join(IN_DIR, f"*{VAR}*_masked*.nc")))
    if not files:
        raise FileNotFoundError(f"No masked files found in {IN_DIR}")

    fields = []
    member_labels = []

    lat_ref = None
    lon_ref = None
    surf_depths = []

    for i, f in enumerate(files, start=1):
        label = parse_member_label(f)
        member_labels.append(label)

        print(f"\n  → Member {i}/{len(files)}: {label}")

        ds = standardize_latlon(xr.open_dataset(f))
        da = ds[VAR]

        da = da.sel(time=slice(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"))

        da_surf, surf_z_m, z_name = select_surface_field(da, ds)
        surf_depths.append(surf_z_m)

        # Ensure standard order
        da_surf = da_surf.transpose("time", "y", "x")

        if REMOVE_TEMPORAL_MEAN:
            da_surf = da_surf - da_surf.mean("time")

        da_surf = da_surf.expand_dims(member=[label])
        fields.append(da_surf)

        if lat_ref is None:
            lat_ref = ds["lat2d"].values
            lon_ref = wrap_lon(ds["lon2d"].values)

        print(f"    Surface depth used: {surf_z_m:.6f} m" if np.isfinite(surf_z_m) else "    No vertical dim found")

        ds.close()

    print("\n▶ Aligning time across members")
    common_time = fields[0]["time"].values
    for s in fields[1:]:
        common_time = np.intersect1d(common_time, s["time"].values)

    common_time = np.sort(common_time)

    combined = xr.concat([s.sel(time=common_time) for s in fields], dim="member")

    n_member = combined.sizes["member"]
    n_time   = combined.sizes["time"]
    ny       = combined.sizes["y"]
    nx       = combined.sizes["x"]

    # TRAIN split on common axis
    n_train = int(np.floor(TRAIN_FRACTION * n_time))
    n_train = max(2, min(n_train, n_time))
    train_time = combined["time"].values[:n_train]

    train_mask = np.zeros(n_time, dtype=bool)
    train_mask[:n_train] = True

    print(f"\n▶ Combined shape: member={n_member}, time={n_time}, y={ny}, x={nx}")
    print(f"▶ TRAIN_FRACTION={TRAIN_FRACTION} -> n_train={n_train}/{n_time} "
          f"(train end = {str(train_time[-1])})")

    # Full and train data
    data_full  = combined.transpose("member", "time", "y", "x").values.astype(float)
    data_train = combined.sel(time=train_time).transpose("member", "time", "y", "x").values.astype(float)

    print("▶ Building constant spatial mask from TRAIN data")
    valid_space = make_space_mask_from_train(data_train)
    n_valid = int(valid_space.sum())

    if n_valid == 0:
        raise ValueError("No spatial points remain after TRAIN valid-space masking")

    print(f"▶ Valid ocean points kept for EOF: {n_valid} / {ny*nx}")

    data2d_train = flatten_valid_space(data_train, valid_space)
    data2d_full  = flatten_valid_space(data_full,  valid_space)

    data2d_train = np.ma.masked_invalid(data2d_train)
    data2d_full  = np.ma.masked_invalid(data2d_full)

    print("▶ Computing EOF weights")
    lat_valid = lat_ref[valid_space]
    weights = np.sqrt(np.clip(np.cos(np.deg2rad(lat_valid)), 0, None))

    print("▶ Solving EOFs on TRAIN only")
    solver = Eof(data2d_train, weights=weights)

    eofs_valid = np.asarray(solver.eofs(neofs=NMODES))
    vf = np.asarray(solver.varianceFraction()[:NMODES])

    print("▶ Projecting FULL period onto TRAIN EOFs")
    pcs_full_flat = np.asarray(solver.projectField(data2d_full, neofs=NMODES))
    pcs = pcs_full_flat.reshape(n_member, n_time, NMODES)

    print("▶ Restoring EOF maps to full grid")
    EOFs = unflatten_eofs_to_fullgrid(eofs_valid, valid_space, ny, nx)

    surface_depth_mean = float(np.nanmean(surf_depths))
    surface_depth_min  = float(np.nanmin(surf_depths))
    surface_depth_max  = float(np.nanmax(surf_depths))

    outfile = os.path.join(OUT_DIR, f"EOF_surface_latlon_{VAR}.nc")
    print(f"▶ Writing {outfile}")

    xr.Dataset(
        {
            "EOF": (("mode", "y", "x"), EOFs),
            "PC":  (("member", "time", "mode"), pcs),
            "variance_fraction": (("mode",), vf),
            "train_mask": (("time",), train_mask),
            "valid_mask": (("y", "x"), valid_space.astype(np.int8)),
        },
        coords={
            "mode": np.arange(1, NMODES + 1),
            "member": np.array(member_labels, dtype=object),
            "time": combined["time"].values,
            "lat": (("y", "x"), lat_ref),
            "lon": (("y", "x"), lon_ref),
        },
        attrs={
            "MODEL": MODEL,
            "VAR": VAR,
            "GRID": "surface_latlon",
            "WEIGHTING": "sqrt(cos(lat))",
            "EOF_FIT": "TRAIN_ONLY",
            "TRAIN_FRACTION": float(TRAIN_FRACTION),
            "TRAIN_START": str(train_time[0]),
            "TRAIN_END": str(train_time[-1]),
            "START_YEAR": int(START_YEAR),
            "END_YEAR": int(END_YEAR),
            "REMOVE_TEMPORAL_MEAN": str(bool(REMOVE_TEMPORAL_MEAN)),
            "SURFACE_LEVEL_SELECTION": "shallowest_available_model_level",
            "SURFACE_DEPTH_MEAN_M": surface_depth_mean,
            "SURFACE_DEPTH_MIN_M": surface_depth_min,
            "SURFACE_DEPTH_MAX_M": surface_depth_max,
        }
    ).to_netcdf(outfile)

    print("✓ Done")

# ============================================================
# RUN
# ============================================================

if __name__ == "__main__":
    print(f"\n=== MODEL={MODEL} | VAR={VAR} ===")
    print(f"TRAIN_FRACTION={TRAIN_FRACTION}")
    print(f"REMOVE_TEMPORAL_MEAN={REMOVE_TEMPORAL_MEAN}")

    run_surface_eof()

    print("\n✅ Surface lat-lon EOF processed successfully.")

## Feature selection

### Top 50 correlation feature 

In [ ]:
# ======================================================
"""
FEATURE RANKING SCRIPT
======================

Produces:
    feature_ranking_pre.csv

The script ranks all EOF-PC predictors by their absolute Pearson correlation
with the AMOC target during the PRE/training period only.

No final_features.csv is generated, because the downstream subset search uses
feature_ranking_pre.csv directly.
"""

import os
import glob
import numpy as np
import pandas as pd
import xarray as xr

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# USER SETTINGS
# ============================================================

MODEL = "CESM2"  # "EC-Earth3", "IPSL-CM6A-LR", "CESM2", "MPI-ESM1-2-LR"
TARGET = "AMOC_45N_ensmean"

MODE = "ensmean"      # "ensmean" or "member"
member_id = None      # e.g. "r1i1p1f1" if MODE="member"

EOF_DIR = (
    f"/data/projects/nckf/frekle/EOF_results/{MODEL}/"
    f"latdepth_sections/Train_period_85pct/"
)

AMOC_FILE = f"/data/users/frekle/AMOC_analysis/AMOC_{MODEL}.nc"

OUTDIR = (
    f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/"
    f"Feature_selection/"
)
os.makedirs(OUTDIR, exist_ok=True)

VARS = ["thetao", "so"]
LON_TAGS = ["W10p0", "W20p0", "W30p0", "W40p0", "W60p0"]

N_MODES = 10
MAX_LAG_ALLOWED = 20

YEAR_START = 1850
YEAR_END = 2014

STANDARDIZE_PC = True


# ============================================================
# HELPER FUNCTIONS
# ============================================================

def extract_years(time_coord):
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        return np.array([int(str(x)[:4]) for x in np.asarray(time_coord)])


def corr_1d(a, b):
    a = np.asarray(a, dtype=float)
    b = np.asarray(b, dtype=float)

    m = np.isfinite(a) & np.isfinite(b)

    if m.sum() < 3:
        return np.nan

    aa = a[m] - np.mean(a[m])
    bb = b[m] - np.mean(b[m])

    denom = np.sqrt(np.sum(aa ** 2) * np.sum(bb ** 2))

    if denom == 0:
        return np.nan

    return float(np.sum(aa * bb) / denom)


def get_train_end():
    files = sorted(glob.glob(os.path.join(EOF_DIR, "EOF_latdepth_*_*.nc")))

    if not files:
        raise FileNotFoundError(f"No EOF files found in {EOF_DIR}")

    ds = xr.open_dataset(files[0])

    if "TRAIN_END" not in ds.attrs:
        ds.close()
        raise KeyError(f"TRAIN_END attribute not found in {files[0]}")

    train_end = int(str(ds.attrs["TRAIN_END"])[:4])

    ds.close()

    return train_end


def load_pc(var, lon_tag, mode="ensmean", member_id=None):
    f = os.path.join(EOF_DIR, f"EOF_latdepth_{var}_{lon_tag}.nc")

    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)

    if "PC" not in ds:
        ds.close()
        raise KeyError(f"'PC' not found in {f}")

    pc = ds["PC"].isel(mode=slice(0, N_MODES))

    if "member" in pc.dims:
        if mode == "ensmean":
            pc = pc.mean("member")

        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when MODE='member'")

            if "member" in pc.coords:
                member_values = [str(m) for m in pc["member"].values]

                if str(member_id) in member_values:
                    pc = pc.sel(member=str(member_id))
                else:
                    ds.close()
                    raise ValueError(f"member_id {member_id} not found in {f}")
            else:
                pc = pc.isel(member=int(member_id))

        else:
            ds.close()
            raise ValueError("MODE must be 'ensmean' or 'member'")

    pc = pc.transpose("time", "mode")

    years = extract_years(pc["time"])

    pc = (
        pc.assign_coords(year=("time", years))
          .swap_dims({"time": "year"})
          .drop_vars("time")
          .astype(float)
          .load()
    )

    ds.close()

    return pc


def load_amoc(mode="ensmean", member_id=None):
    ds = xr.open_dataset(AMOC_FILE)

    if TARGET not in ds:
        ds.close()
        raise KeyError(
            f"Target '{TARGET}' not found in {AMOC_FILE}. "
            f"Available variables: {list(ds.data_vars)}"
        )

    y = ds[TARGET].squeeze()

    if "member" in y.dims:
        if mode == "ensmean":
            y = y.mean("member")

        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when MODE='member'")

            if "member" in y.coords:
                member_values = [str(m) for m in y["member"].values]

                if str(member_id) in member_values:
                    y = y.sel(member=str(member_id))
                else:
                    ds.close()
                    raise ValueError(f"member_id {member_id} not found in AMOC file")
            else:
                if isinstance(member_id, str) and member_id.startswith("r"):
                    member_index = int(member_id.split("i")[0].replace("r", "")) - 1
                else:
                    member_index = int(member_id)

                y = y.isel(member=member_index)

        else:
            ds.close()
            raise ValueError("MODE must be 'ensmean' or 'member'")

    if "year" not in y.dims:
        years = extract_years(y["time"])
        y = (
            y.assign_coords(year=("time", years))
             .swap_dims({"time": "year"})
             .drop_vars("time")
        )

    y = y.astype(float)

    ds.close()

    return y


def get_all_members():
    required_files = [
        os.path.join(EOF_DIR, f"EOF_latdepth_{var}_{lon}.nc")
        for var in VARS
        for lon in LON_TAGS
    ]

    member_sets = []
    member_order = None

    for f in required_files:
        if not os.path.exists(f):
            raise FileNotFoundError(f"Missing EOF file: {f}")

        ds = xr.open_dataset(f)

        if "PC" not in ds:
            ds.close()
            raise KeyError(f"'PC' not found in {f}")

        pc = ds["PC"]

        if "member" not in pc.dims:
            ds.close()
            raise ValueError(f"PC has no member dimension in {f}")

        if "member" in pc.coords:
            members = [str(m) for m in pc["member"].values]
        else:
            members = [str(i) for i in range(pc.sizes["member"])]

        if member_order is None:
            member_order = members

        member_sets.append(set(members))

        ds.close()

    common_members = set.intersection(*member_sets)
    members_to_run = [m for m in member_order if m in common_members]

    return members_to_run


def run_feature_ranking_for_member(mem):
    if MODE == "member":
        print(f"\nRUNNING MEMBER: {mem}")
        outdir_member = os.path.join(OUTDIR, str(mem))
    else:
        print("\nRUNNING ENSEMBLE MEAN")
        outdir_member = OUTDIR

    os.makedirs(outdir_member, exist_ok=True)

    amoc = load_amoc(mode=MODE, member_id=mem)

    pc_dict = {
        (var, lon): load_pc(var, lon, mode=MODE, member_id=mem)
        for var in VARS
        for lon in LON_TAGS
    }

    years = amoc["year"].values.astype(int)

    for pc in pc_dict.values():
        years = np.intersect1d(years, pc["year"].values.astype(int))

    years = years[(years >= YEAR_START) & (years <= YEAR_END)]
    years.sort()

    if len(years) == 0:
        raise ValueError(f"No overlapping years found for member {mem}")

    y = amoc.sel(year=years).values.astype(float)

    pc_np = {
        key: pc.sel(year=years).values.astype(float)
        for key, pc in pc_dict.items()
    }

    idx_pre = np.where(years <= TRAIN_END_YEAR)[0]
    idx_post = np.where(years > TRAIN_END_YEAR)[0]

    if len(idx_pre) == 0:
        raise ValueError(f"PRE period is empty for member {mem}")

    print(f"PRE:  {years[idx_pre[0]]}–{years[idx_pre[-1]]}")

    if len(idx_post) > 0:
        print(f"POST: {years[idx_post[0]]}–{years[idx_post[-1]]}")

    y_pre = y[idx_pre]

    rows = []

    for var in VARS:
        for lon in LON_TAGS:
            X_pre = pc_np[(var, lon)][idx_pre, :]

            if STANDARDIZE_PC:
                X_pre = (
                    X_pre - np.nanmean(X_pre, axis=0)
                ) / (
                    np.nanstd(X_pre, axis=0) + 1e-12
                )

            for lag in range(MAX_LAG_ALLOWED + 1):
                if lag >= len(y_pre):
                    continue

                t = np.arange(lag, len(y_pre))

                for mode0 in range(N_MODES):
                    c = corr_1d(
                        X_pre[t - lag, mode0],
                        y_pre[t]
                    )

                    if np.isfinite(c):
                        rows.append({
                            "var": var,
                            "lon_tag": lon,
                            "mode": mode0 + 1,
                            "lag": lag,
                            "corr": c,
                            "abs_corr": abs(c),
                        })

    df_rank = (
        pd.DataFrame(rows)
          .sort_values("abs_corr", ascending=False)
          .reset_index(drop=True)
    )

    rank_file = os.path.join(outdir_member, "feature_ranking_pre.csv")
    df_rank.to_csv(rank_file, index=False)

    print(f"Saved: {rank_file}")
    print(f"Number of ranked features: {len(df_rank)}")

    print("\nTop 10 ranked features:")
    print(df_rank.head(10).to_string(index=False))


# ============================================================
# MAIN
# ============================================================

print("Loading setup...")

TRAIN_END_YEAR = get_train_end()
print(f"TRAIN_END_YEAR: {TRAIN_END_YEAR}")

if MODE == "member" and member_id is None:
    members_to_run = get_all_members()
    print(f"Running all members: {len(members_to_run)}")
else:
    members_to_run = [member_id]

for mem in members_to_run:
    run_feature_ranking_for_member(mem)

print("\nFeature ranking completed.")

### Subset search

In [ ]:
# ==============================================
# SUBSET SEARCH FROM PRE-RANKED FEATURES
# ==============================================
"""
Purpose
-------
Run exhaustive subset search directly from feature_ranking_pre.csv,
without any spike analysis or curve-based candidate selection.

This version evaluates each subset only over the period where all of its
lagged predictors are available. Therefore, the reconstruction window for a
given subset starts at:

    start_lag_subset = max(lag of features in that subset)

This makes the subset evaluation internally consistent with the actual lagged
predictor structure.

Inputs
------
- feature_ranking_pre.csv
- EOF_latdepth_{var}_{lon}.nc
- AMOC_{MODEL}.nc

Outputs
-------
- best_subsets_by_k.csv
- all_subsets_scored.csv   (optional)
- summary_best_subsets.txt

Notes
-----
- Candidate features are taken directly from the top rows of feature_ranking_pre.csv
- Lag filtering can still be applied (for precursor-focused analyses)
- Selection can still be based on TRAIN or TEST R²
"""

import os
import glob
import itertools
import numpy as np
import pandas as pd
import xarray as xr

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# USER SETTINGS
# ============================================================
MODEL  = "CESM2"   # "EC-Earth3", "IPSL-CM6A-LR", "CESM2", "MPI-ESM1-2-LR"
TARGET = "AMOC_45N_ensmean"

# Choose one:
MODE = "ensmean"
member_id = None

# Example for member mode:
# MODE = "member"
# member_id = "r1i1p1f1"

# Choose AMOC variable:
#   "normal" -> use TARGET as-is
#   "smooth" -> map to AMOC_26N_smooth / AMOC_45N_smooth
AMOC_VARIANT = "normal"   # "normal" or "smooth"

EOF_DIR   = f"/data/projects/nckf/frekle/EOF_results/{MODEL}/latdepth_sections/Train_period_85pct/"
AMOC_FILE = f"/data/users/frekle/AMOC_analysis/AMOC_{MODEL}.nc"
RANK_CSV  = f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Feature_selection/feature_ranking_pre.csv"

# Candidate selection from ranking
N_CANDIDATES = 50   # use top N ranked features directly

# Subset selection criterion
SELECT_ON   = "train"   # "train" or "test"
TIEBREAK_ON = "test"    # "train" or "test"

# Precursor lag constraint (applied to candidate pool only)
MIN_LAG = 3
MAX_LAG = None   # e.g. 20, or None

# Detrending (TRAIN-fit)
DETREND_Y = False
DETREND_X = False

OUTDIR = (
    f"/data/users/frekle/Final_figures/{MODEL}/{TARGET}/Feature_selection/"
    f"Detrended_{DETREND_Y}/Selected_on_{SELECT_ON}/amoc_variant_{AMOC_VARIANT}/"
    f"lag_policy_{MIN_LAG}minlag_{MAX_LAG if MAX_LAG is not None else 'None'}max"
)
os.makedirs(OUTDIR, exist_ok=True)

# PC setup
N_MODES = 10

# Ridge hyperparameters
ALPHA = 1.0

# Subset search settings
MAX_FEATURES_TO_USE = 3
SAVE_ALL_SUBSETS = True


# ============================================================
# Helpers
# ============================================================
def extract_years(time_coord):
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        t = np.asarray(time_coord)
        return np.array([int(str(x)[:4]) for x in t], dtype=int)


def load_pc_latdepth(var, lon_tag, n_modes, eof_dir, mode="ensmean", member_id=None):
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon_tag}.nc")
    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)
    if "PC" not in ds:
        ds.close()
        raise KeyError(f"'PC' not found in {f}")

    PC = ds["PC"].isel(mode=slice(0, int(n_modes)))

    if "member" in PC.dims:
        if mode == "ensmean":
            PC = PC.mean("member")
        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when mode='member'")
            if "member" in PC.coords and member_id in PC["member"].values:
                PC = PC.sel(member=member_id)
            else:
                PC = PC.isel(member=int(member_id))
        else:
            ds.close()
            raise ValueError("mode must be 'ensmean' or 'member'")

    PC = PC.transpose("time", "mode")
    years = extract_years(PC["time"])
    PC = PC.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    ds.close()
    return PC.astype(float)


def _infer_amoc_lat_from_target(target_name: str):
    if "26N" in target_name:
        return "26N"
    if "45N" in target_name:
        return "45N"
    raise ValueError(f"Could not infer latitude tag (26N/45N) from TARGET='{target_name}'")


def resolve_amoc_variable(target: str, amoc_variant: str):
    amoc_variant = str(amoc_variant).strip().lower()
    if amoc_variant == "normal":
        return target
    if amoc_variant == "smooth":
        lat = _infer_amoc_lat_from_target(target)
        return f"AMOC_{lat}_smooth"
    raise ValueError("AMOC_VARIANT must be 'normal' or 'smooth'")


def load_amoc(target, amoc_file, mode="ensmean", member_id=None, amoc_variant="normal"):
    varname = resolve_amoc_variable(target, amoc_variant)

    ds = xr.open_dataset(amoc_file)
    if varname not in ds:
        ds.close()
        raise KeyError(f"AMOC var '{varname}' not found in {amoc_file}. Available: {list(ds.data_vars)}")

    y = ds[varname].squeeze()

    if "year" in y.dims:
        am = y
    else:
        years = extract_years(y["time"])
        am = y.assign_coords(year=("time", years)).swap_dims({"time": "year"}).drop_vars("time")

    am = am.astype(float)

    if "member" in am.dims:
        if mode == "ensmean":
            am = am.mean("member")
        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when mode='member'")
            if "member" in am.coords and member_id in am["member"].values:
                am = am.sel(member=member_id)
            else:
                am = am.isel(member=int(member_id))
        else:
            ds.close()
            raise ValueError("mode must be 'ensmean' or 'member'")

    ds.close()
    return am.squeeze()


def infer_train_end_year_from_any_eof(eof_dir, prefer=("so", "W30p0")):
    var, lon = prefer
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon}.nc")
    if not os.path.exists(f):
        hits = sorted(glob.glob(os.path.join(eof_dir, "EOF_latdepth_*_*.nc")))
        if not hits:
            raise FileNotFoundError(f"No EOF files found in {eof_dir}")
        f = hits[0]

    ds = xr.open_dataset(f)

    if "TRAIN_END" in ds.attrs:
        y = int(str(ds.attrs["TRAIN_END"])[:4])
        ds.close()
        return y

    if "train_mask" in ds:
        tm = ds["train_mask"].values.astype(bool)
        if tm.any():
            t_last = ds["time"].values[np.where(tm)[0][-1]]
            ds.close()
            return int(str(np.datetime64(t_last))[:4])

    ds.close()
    raise RuntimeError(f"Could not infer TRAIN_END year from {f}")


def fit_linear_trend(train_years, train_series):
    x = np.asarray(train_years, float)
    y = np.asarray(train_series, float)
    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    if len(x) < 2:
        return 0.0, float(np.nanmean(y))

    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]
    return float(a), float(b)


def detrend_with_train_fit(all_years, all_series, train_mask):
    all_years = np.asarray(all_years, float)
    all_series = np.asarray(all_series, float)
    a, b = fit_linear_trend(all_years[train_mask], all_series[train_mask])
    trend = a * all_years + b
    return all_series - trend, (a, b)


def pearson_corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)
    m = np.isfinite(a) & np.isfinite(b)
    if m.sum() < 2:
        return np.nan
    a = a[m]
    b = b[m]
    sa = a.std()
    sb = b.std()
    if sa == 0 or sb == 0:
        return np.nan
    return float(np.corrcoef(a, b)[0, 1])


def finite_rows_mask(Y, X):
    Y = np.asarray(Y, float)
    X = np.asarray(X, float)
    m = np.isfinite(Y)
    if X.ndim == 1:
        m = m & np.isfinite(X)
    else:
        m = m & np.all(np.isfinite(X), axis=1)
    return m


def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)
    if len(y_true) < 2:
        return np.nan
    return float(r2_score(y_true, y_pred))


def choose_better(curr, best, select_on="train", tiebreak_on="test"):
    if select_on not in ("train", "test"):
        raise ValueError("SELECT_ON must be 'train' or 'test'")
    if tiebreak_on not in ("train", "test"):
        raise ValueError("TIEBREAK_ON must be 'train' or 'test'")

    key_main = "r2_train" if select_on == "train" else "r2_test"
    key_tie  = "r2_train" if tiebreak_on == "train" else "r2_test"

    if curr[key_main] > best[key_main]:
        return True
    if np.isclose(curr[key_main], best[key_main]) and curr[key_tie] > best[key_tie]:
        return True
    return False


def load_ranked_features(csv_path):
    """
    Accepts either lon_tag or lon.
    Expects columns: var, (lon_tag|lon), mode, lag
    mode is assumed 1-based in file and converted to 0-based internally.
    """
    df = pd.read_csv(csv_path)

    if "lon" not in df.columns and "lon_tag" in df.columns:
        df = df.rename(columns={"lon_tag": "lon"})
    if "lon" not in df.columns:
        raise KeyError(f"Missing lon/lon_tag in {csv_path}. Columns: {list(df.columns)}")

    for c in ["var", "lon", "mode", "lag"]:
        if c not in df.columns:
            raise KeyError(f"Missing column '{c}' in {csv_path}. Columns: {list(df.columns)}")

    df["var"] = df["var"].astype(str).str.strip()
    df["lon"] = df["lon"].astype(str).str.strip()
    df["mode0"] = df["mode"].astype(int) - 1
    df["lag"] = df["lag"].astype(int)

    feats = [(r["var"], r["lon"], int(r["mode0"]), int(r["lag"])) for _, r in df.iterrows()]
    return feats, df


def filter_feats_by_lag(feats, min_lag=0, max_lag=None):
    out = []
    for (v, lon, m0, lag) in feats:
        if lag < int(min_lag):
            continue
        if max_lag is not None and lag > int(max_lag):
            continue
        out.append((v, lon, m0, lag))
    return out


def build_XY_for_subset(subset_feats, years, y_amoc, pc_np, train_end_year, detrend_y=False, detrend_x=False):
    """
    Build X and Y for one subset using subset-specific start lag:
        start_lag_subset = max(lag in subset_feats)
    """
    if len(subset_feats) == 0:
        raise ValueError("subset_feats is empty")

    start_lag_subset = int(max(lag for *_, lag in subset_feats))
    used = np.arange(start_lag_subset, len(years), dtype=int)

    if len(used) == 0:
        raise RuntimeError("No valid time indices after applying subset lag window.")

    years_used = years[used]
    Y = y_amoc[used].astype(float)

    X = np.empty((len(used), len(subset_feats)), dtype=float)
    for j, (var, lon, m0, lag) in enumerate(subset_feats):
        X[:, j] = pc_np[(var, lon)][used - int(lag), int(m0)]

    idx_tr = np.where(years_used <= train_end_year)[0]
    idx_te = np.where(years_used > train_end_year)[0]

    if len(idx_tr) == 0:
        raise RuntimeError("Train period is empty after subset-specific windowing.")
    if len(idx_te) == 0:
        raise RuntimeError("Test period is empty after subset-specific windowing.")

    train_mask_used = np.zeros(len(years_used), dtype=bool)
    train_mask_used[idx_tr] = True

    if detrend_y:
        Y_dt, (ay, by) = detrend_with_train_fit(years_used, Y, train_mask_used)
    else:
        Y_dt = Y.copy()
        ay, by = 0.0, 0.0

    if detrend_x:
        X_dt = X.copy()
        for j in range(X.shape[1]):
            X_dt[:, j], _ = detrend_with_train_fit(years_used, X[:, j], train_mask_used)
    else:
        X_dt = X.copy()

    return {
        "start_lag_subset": start_lag_subset,
        "used": used,
        "years_used": years_used,
        "Y_dt": Y_dt,
        "X_dt": X_dt,
        "idx_tr": idx_tr,
        "idx_te": idx_te,
        "trend_y": (ay, by),
    }


# ============================================================
# 1) Build candidate feature list directly from ranking
# ============================================================
feats_ranked, _ = load_ranked_features(RANK_CSV)

cand_feats = feats_ranked[:N_CANDIDATES]
cand_feats = filter_feats_by_lag(cand_feats, min_lag=MIN_LAG, max_lag=MAX_LAG)

# remove duplicates while preserving order
seen = set()
cand_feats_unique = []
for ft in cand_feats:
    if ft not in seen:
        cand_feats_unique.append(ft)
        seen.add(ft)
cand_feats = cand_feats_unique

M = len(cand_feats)
if M == 0:
    raise RuntimeError(
        f"No candidate features left after lag filtering: MIN_LAG={MIN_LAG}, MAX_LAG={MAX_LAG}"
    )

print("\n==============================")
print("RANKING-BASED CANDIDATE FEATURES")
print("==============================")
print(f"Top ranked features used: {N_CANDIDATES}")
print(f"Candidate features after lag filter: {M}")
print(f"MIN_LAG filter: >= {MIN_LAG}" + (f", <= {MAX_LAG}" if MAX_LAG is not None else ""))
for i, (v, lon, m0, lag) in enumerate(cand_feats, 1):
    print(f"  C{i}: {v} {lon} EOF{m0+1} lag{lag}")

KMAX = min(MAX_FEATURES_TO_USE, M)


# ============================================================
# 2) Load AMOC + PCs needed
# ============================================================
TRAIN_END_YEAR = infer_train_end_year_from_any_eof(EOF_DIR, prefer=("so", "W30p0"))
print("\n✅ TRAIN_END_YEAR inferred from EOF files:", TRAIN_END_YEAR)

amoc_var_used = resolve_amoc_variable(TARGET, AMOC_VARIANT)
print(f"✅ AMOC variable used: {amoc_var_used}   (AMOC_VARIANT={AMOC_VARIANT})")

amoc = load_amoc(TARGET, AMOC_FILE, mode=MODE, member_id=member_id, amoc_variant=AMOC_VARIANT)

needed_pairs = sorted(set((v, lon) for (v, lon, _, _) in cand_feats))
pc_dict = {
    (v, lon): load_pc_latdepth(v, lon, N_MODES, EOF_DIR, mode=MODE, member_id=member_id)
    for (v, lon) in needed_pairs
}

common_years = amoc["year"].values.astype(int)
for da in pc_dict.values():
    common_years = np.intersect1d(common_years, da["year"].values.astype(int))

years = np.asarray(common_years, int)
years.sort()

if len(years) == 0:
    raise RuntimeError("No overlapping years found between AMOC and PCs.")

y_amoc = amoc.sel(year=years).values.astype(float)
pc_np = {k: v.sel(year=years).values.astype(float) for k, v in pc_dict.items()}

print("\nOverlap years:")
print(f"{years[0]} – {years[-1]}   (n={len(years)})")


# ============================================================
# 3) Exhaustive subset search
# ============================================================
all_rows = []
best_by_k = []

print("\n==============================")
print("SUBSET SEARCH")
print("==============================")
print(f"Selecting BEST subsets by: {SELECT_ON.upper()} R² (tie-break: {TIEBREAK_ON.upper()} R²)")
print(f"Target AMOC: {amoc_var_used}")
print(f"ALPHA={ALPHA}  DETREND_Y={DETREND_Y}  DETREND_X={DETREND_X}")
print(f"KMAX={KMAX}  Candidates={M}")
print("Windowing: subset-specific start lag = max lag within each tested subset\n")

for k in range(1, KMAX + 1):
    best = {
        "k": k,
        "r2_train": -np.inf,
        "corr_train": np.nan,
        "r2_test": -np.inf,
        "corr_test": np.nan,
        "subset_idx": None,
        "start_lag_subset": np.nan,
        "years_used_start": np.nan,
        "years_used_end": np.nan,
        "n_train": np.nan,
        "n_test": np.nan,
    }

    for subset_idx in itertools.combinations(range(M), k):
        subset_idx = tuple(subset_idx)
        subset_feats = [cand_feats[i] for i in subset_idx]

        try:
            built = build_XY_for_subset(
                subset_feats,
                years,
                y_amoc,
                pc_np,
                train_end_year=TRAIN_END_YEAR,
                detrend_y=DETREND_Y,
                detrend_x=DETREND_X,
            )
        except RuntimeError:
            continue

        Xk = built["X_dt"]
        Y_dt = built["Y_dt"]
        years_used = built["years_used"]
        idx_tr = built["idx_tr"]
        idx_te = built["idx_te"]
        start_lag_subset = built["start_lag_subset"]

        mdl = make_pipeline(StandardScaler(), Ridge(alpha=float(ALPHA)))

        Ytr = Y_dt[idx_tr]
        Xtr = Xk[idx_tr]
        m_tr = finite_rows_mask(Ytr, Xtr)

        Yte = Y_dt[idx_te]
        Xte = Xk[idx_te]
        m_te = finite_rows_mask(Yte, Xte)

        if m_tr.sum() < 3:
            continue
        if m_te.sum() < 2:
            continue

        mdl.fit(Xtr[m_tr], Ytr[m_tr])

        pred_tr = mdl.predict(Xtr[m_tr])
        r2_tr = safe_r2(Ytr[m_tr], pred_tr)
        c_tr = pearson_corr(Ytr[m_tr], pred_tr)

        pred_te = mdl.predict(Xte[m_te])
        r2_te = safe_r2(Yte[m_te], pred_te)
        c_te = pearson_corr(Yte[m_te], pred_te)

        if not np.isfinite(r2_tr) or not np.isfinite(r2_te):
            continue

        curr = {
            "r2_train": r2_tr,
            "corr_train": c_tr,
            "r2_test": r2_te,
            "corr_test": c_te,
            "subset_idx": subset_idx,
            "start_lag_subset": start_lag_subset,
            "years_used_start": int(years_used[0]),
            "years_used_end": int(years_used[-1]),
            "n_train": int(len(idx_tr)),
            "n_test": int(len(idx_te)),
        }

        if SAVE_ALL_SUBSETS:
            subset_str = " | ".join(
                [f"{v} {lon} EOF{m0+1} lag{lag}" for (v, lon, m0, lag) in subset_feats]
            )
            all_rows.append({
                "k": k,
                "r2_train": r2_tr,
                "corr_train": c_tr,
                "r2_test": r2_te,
                "corr_test": c_te,
                "subset_idx": ",".join(map(str, subset_idx)),
                "subset_features": subset_str,
                "start_lag_subset": start_lag_subset,
                "years_used_start": int(years_used[0]),
                "years_used_end": int(years_used[-1]),
                "n_train": int(len(idx_tr)),
                "n_test": int(len(idx_te)),
                "selected_by": SELECT_ON,
                "amoc_variant": AMOC_VARIANT,
                "amoc_var_used": amoc_var_used,
            })

        if best["subset_idx"] is None or choose_better(curr, best, select_on=SELECT_ON, tiebreak_on=TIEBREAK_ON):
            best.update(curr)

    if best["subset_idx"] is None:
        print(f"No valid subset found for k={k}")
        continue

    subset_feats = [cand_feats[i] for i in best["subset_idx"]]
    subset_str = " | ".join([f"{v} {lon} EOF{m0+1} lag{lag}" for (v, lon, m0, lag) in subset_feats])

    best_by_k.append({
        "k": k,
        "r2_train": best["r2_train"],
        "corr_train": best["corr_train"],
        "r2_test": best["r2_test"],
        "corr_test": best["corr_test"],
        "subset_idx": ",".join(map(str, best["subset_idx"])),
        "subset_features": subset_str,
        "start_lag_subset": best["start_lag_subset"],
        "years_used_start": best["years_used_start"],
        "years_used_end": best["years_used_end"],
        "n_train": best["n_train"],
        "n_test": best["n_test"],
        "selected_by": SELECT_ON,
        "amoc_variant": AMOC_VARIANT,
        "amoc_var_used": amoc_var_used,
    })

    print(
        f"BEST k={k} (selected on {SELECT_ON}): "
        f"Train R²={best['r2_train']:.4f} corr={best['corr_train']:.4f} | "
        f"Test R²={best['r2_test']:.4f} corr={best['corr_test']:.4f}"
    )
    print(
        f"  start_lag_subset={best['start_lag_subset']} | "
        f"years_used={best['years_used_start']}–{best['years_used_end']} | "
        f"n_train={best['n_train']} n_test={best['n_test']}"
    )
    for (v, lon, m0, lag) in subset_feats:
        print(f"  - {v:6s} {lon:6s} EOF{m0+1:<2d} lag{lag}")
    print("")


# ============================================================
# 4) Save results
# ============================================================
best_df = pd.DataFrame(best_by_k)
best_csv = os.path.join(OUTDIR, "best_subsets_by_k.csv")
best_df.to_csv(best_csv, index=False)
print("\n✅ Saved best subsets by k:", best_csv)

if SAVE_ALL_SUBSETS:
    sort_main = "r2_train" if SELECT_ON == "train" else "r2_test"
    sort_tie  = "r2_train" if TIEBREAK_ON == "train" else "r2_test"

    all_df = (
        pd.DataFrame(all_rows)
        .sort_values(["k", sort_main, sort_tie], ascending=[True, False, False])
        .reset_index(drop=True)
    )

    all_csv = os.path.join(OUTDIR, "all_subsets_scored.csv")
    all_df.to_csv(all_csv, index=False)
    print("✅ Saved all subset scores:", all_csv)

summary_txt = os.path.join(OUTDIR, "summary_best_subsets.txt")
with open(summary_txt, "w") as f:
    f.write(f"MODEL={MODEL}\n")
    f.write(f"TARGET={TARGET}\n")
    f.write(f"MODE={MODE}\n")
    f.write(f"AMOC_VARIANT={AMOC_VARIANT}\n")
    f.write(f"AMOC_VAR_USED={amoc_var_used}\n")
    f.write(f"TRAIN_END_YEAR={TRAIN_END_YEAR}\n")
    f.write(f"ALPHA={ALPHA}\n")
    f.write(f"DETREND_Y={DETREND_Y}\n")
    f.write(f"DETREND_X={DETREND_X}\n")
    f.write(f"RANK_CSV={RANK_CSV}\n")
    f.write(f"N_CANDIDATES={N_CANDIDATES}\n")
    f.write(f"SELECT_ON={SELECT_ON}\n")
    f.write(f"TIEBREAK_ON={TIEBREAK_ON}\n")
    f.write(f"MIN_LAG={MIN_LAG}\n")
    f.write(f"MAX_LAG={MAX_LAG}\n")
    f.write("WINDOWING=subset-specific max lag\n")

    f.write("\nBest subsets by k:\n")
    for row in best_by_k:
        f.write(
            f"\nK={row['k']}  "
            f"R2_train={row['r2_train']:.6f}  corr_train={row['corr_train']:.6f}  "
            f"R2_test={row['r2_test']:.6f}  corr_test={row['corr_test']:.6f}\n"
        )
        f.write(
            f"start_lag_subset={row['start_lag_subset']}  "
            f"years_used={row['years_used_start']}-{row['years_used_end']}  "
            f"n_train={row['n_train']}  n_test={row['n_test']}\n"
        )
        f.write(row["subset_features"] + "\n")

print("✅ Saved summary:", summary_txt)
print("\nDONE.")

## Reconstructing the AMOC through Ridge Regression

In [ ]:
# ============================================================
# Reconstructing the AMOC through Ridge Regression
# ============================================================

import os
import re
import glob
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.metrics import r2_score
from matplotlib.ticker import MultipleLocator

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# USER SETTINGS
# ============================================================

MODELS = ["CESM2", "EC-Earth3", "IPSL-CM6A-LR", "MPI-ESM1-2-LR"]

TARGET = "AMOC_45N_ensmean"
LATITUDE = "45°N"

MODE = "ensmean"      # "ensmean" or "member"
member_id = None

AMOC_VARIANT = "normal"  # "normal" or "smooth"

# Toggle detrending here
DETREND_Y = False
DETREND_X = False

SELECT_ON = "train"
MIN_LAG = 3
MAX_LAG = None

K_MAX = 3
N_MODES = 10
ALPHA_FIXED = 1.0

BASE_EOF_DIR = "/data/projects/nckf/frekle/EOF_results"
BASE_AMOC_DIR = "/data/users/frekle/AMOC_analysis"

OUT_BASE = "/data/users/frekle/Final_figures/Notebook_reconstruction_only"
os.makedirs(OUT_BASE, exist_ok=True)

FIGSIZE = (13, 5)
LABEL_FS = 16
TICK_FS = 14
LEGEND_FS = 12
LW_AMOC = 2.8
LW_RECON = 2.0

COLOR_MAP = {
    1: "tab:orange",
    2: "tab:green",
    3: "tab:blue",
}


# ============================================================
# HELPERS
# ============================================================

def extract_years(time_coord):
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        return np.array([int(str(x)[:4]) for x in np.asarray(time_coord)], dtype=int)


def infer_train_end_year_from_any_eof(eof_dir, prefer=("so", "W30p0")):
    var, lon = prefer
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon}.nc")

    if not os.path.exists(f):
        hits = sorted(glob.glob(os.path.join(eof_dir, "EOF_latdepth_*_*.nc")))
        if not hits:
            raise FileNotFoundError(f"No EOF files found in {eof_dir}")
        f = hits[0]

    ds = xr.open_dataset(f)

    if "TRAIN_END" in ds.attrs:
        y = int(str(ds.attrs["TRAIN_END"])[:4])
        ds.close()
        return y

    if "train_mask" in ds:
        tm = ds["train_mask"].values.astype(bool)
        if tm.any():
            t_last = ds["time"].values[np.where(tm)[0][-1]]
            ds.close()
            return int(str(np.datetime64(t_last))[:4])

    ds.close()
    raise RuntimeError(f"Could not infer TRAIN_END year from {f}")


def infer_amoc_lat_from_target(target_name):
    if "26N" in target_name:
        return "26N"
    if "45N" in target_name:
        return "45N"
    raise ValueError(f"Could not infer latitude from TARGET={target_name}")


def resolve_amoc_variable(target, amoc_variant):
    if amoc_variant == "normal":
        return target
    if amoc_variant == "smooth":
        lat = infer_amoc_lat_from_target(target)
        return f"AMOC_{lat}_smooth"
    raise ValueError("AMOC_VARIANT must be 'normal' or 'smooth'")


def load_amoc(model, target, mode="ensmean", member_id=None, amoc_variant="normal"):
    amoc_file = os.path.join(BASE_AMOC_DIR, f"AMOC_{model}.nc")
    varname = resolve_amoc_variable(target, amoc_variant)

    ds = xr.open_dataset(amoc_file)

    if varname not in ds.data_vars:
        available = list(ds.data_vars)
        ds.close()
        raise KeyError(f"{varname} not found in {amoc_file}. Available: {available}")

    y = ds[varname].squeeze()

    if "year" in y.dims:
        yy = y
        if "time" in yy.coords:
            yy = yy.drop_vars("time")
    else:
        years_tmp = extract_years(y["time"])
        yy = (
            y.assign_coords(year=("time", years_tmp))
             .swap_dims({"time": "year"})
             .drop_vars("time")
        )

    yy = yy.astype(float)

    if "member" in yy.dims:
        if mode == "ensmean":
            yy = yy.mean("member")
        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when MODE='member'")

            if "member" in yy.coords and member_id in yy["member"].values:
                yy = yy.sel(member=member_id)
            else:
                yy = yy.isel(member=int(member_id))
        else:
            ds.close()
            raise ValueError("MODE must be 'ensmean' or 'member'")

    ds.close()
    return yy.squeeze()


def load_pc_latdepth(model, var, lon_tag, n_modes, mode="ensmean", member_id=None):
    eof_dir = os.path.join(BASE_EOF_DIR, model, "latdepth_sections", "Train_period_85pct")
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon_tag}.nc")

    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)

    if "PC" not in ds:
        ds.close()
        raise KeyError(f"'PC' not found in {f}")

    pc = ds["PC"].isel(mode=slice(0, int(n_modes)))

    if "member" in pc.dims:
        if mode == "ensmean":
            pc = pc.mean("member")
        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when MODE='member'")

            if "member" in pc.coords and member_id in pc["member"].values:
                pc = pc.sel(member=member_id)
            else:
                pc = pc.isel(member=int(member_id))
        else:
            ds.close()
            raise ValueError("MODE must be 'ensmean' or 'member'")

    pc = pc.transpose("time", "mode")
    years_tmp = extract_years(pc["time"])

    pc = (
        pc.assign_coords(year=("time", years_tmp))
          .swap_dims({"time": "year"})
          .drop_vars("time")
          .astype(float)
    )

    ds.close()
    return pc


def parse_features_string(s):
    feats = []

    for part in str(s).split("|"):
        part = part.strip()
        m = re.search(r"(\w+)\s+(W\d+p\d+)\s+EOF(\d+)\s+lag(\d+)", part)

        if not m:
            continue

        var = m.group(1)
        lon = m.group(2)
        mode0 = int(m.group(3)) - 1
        lag = int(m.group(4))

        feats.append((var, lon, mode0, lag))

    out = []
    seen = set()

    for ft in feats:
        if ft not in seen:
            out.append(ft)
            seen.add(ft)

    return out


def fit_linear_trend(train_years, train_series):
    x = np.asarray(train_years, float)
    y = np.asarray(train_series, float)

    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    if len(x) < 2:
        return 0.0, float(np.nanmean(y))

    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]

    return float(a), float(b)


def detrend_with_train_fit(all_years, all_series, train_mask):
    all_years = np.asarray(all_years, float)
    all_series = np.asarray(all_series, float)

    a, b = fit_linear_trend(
        all_years[train_mask],
        all_series[train_mask]
    )

    return all_series - (a * all_years + b), (a, b)


def pearson_corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)

    m = np.isfinite(a) & np.isfinite(b)

    if m.sum() < 3:
        return np.nan

    a = a[m]
    b = b[m]

    if a.std() == 0 or b.std() == 0:
        return np.nan

    return float(np.corrcoef(a, b)[0, 1])


def finite_rows_mask(y, X):
    y = np.asarray(y, float)
    X = np.asarray(X, float)

    m = np.isfinite(y)

    if X.ndim == 1:
        m = m & np.isfinite(X)
    else:
        m = m & np.all(np.isfinite(X), axis=1)

    return m


def safe_r2(y_true, y_pred):
    y_true = np.asarray(y_true, float)
    y_pred = np.asarray(y_pred, float)

    m = np.isfinite(y_true) & np.isfinite(y_pred)

    if m.sum() < 2:
        return np.nan

    return float(r2_score(y_true[m], y_pred[m]))


def pretty_var_name(v):
    if str(v).lower() == "thetao":
        return "Temperature"
    if str(v).lower() == "so":
        return "Salinity"
    return str(v)


def pretty_lon_name(lon_tag):
    m = re.match(r"W(\d+)p(\d+)", str(lon_tag))
    if m:
        whole = int(m.group(1))
        dec = int(m.group(2))
        if dec == 0:
            return f"{whole}°W"
        return f"{whole}.{dec}°W"
    return str(lon_tag)


def human_readable_feature(ft):
    var, lon, mode0, lag = ft
    return f"{pretty_var_name(var)}, {pretty_lon_name(lon)}, EOF{mode0 + 1}, lag {lag} yr"


def get_best_csv_path(model):
    maxlag_tag = MAX_LAG if MAX_LAG is not None else "None"

    return (
        f"/data/users/frekle/Final_figures/{model}/{TARGET}/Feature_selection/"
        f"Detrended_{DETREND_Y}/Selected_on_{SELECT_ON}/amoc_variant_{AMOC_VARIANT}/"
        f"lag_policy_{MIN_LAG}minlag_{maxlag_tag}max/best_subsets_by_k.csv"
    )


def get_model_outdir(model):
    outdir = os.path.join(
        OUT_BASE,
        model,
        TARGET,
        f"Detrended_{DETREND_Y}",
        f"amoc_variant_{AMOC_VARIANT}",
        f"minlag_{MIN_LAG}"
    )

    os.makedirs(outdir, exist_ok=True)
    return outdir


def build_XY_for_feats(feats, years_arr, y_amoc, pc_np, train_end_year):
    if len(feats) == 0:
        raise ValueError("No features provided.")

    maxlag = int(max(lag for *_, lag in feats))

    used = np.arange(maxlag, len(years_arr), dtype=int)
    years_used = years_arr[used]

    Y = y_amoc[used].astype(float)

    X = np.empty((len(used), len(feats)), float)

    for j, (var, lon, mode0, lag) in enumerate(feats):
        X[:, j] = pc_np[(var, lon)][used - int(lag), int(mode0)]

    idx_tr = np.where(years_used <= train_end_year)[0]
    idx_te = np.where(years_used > train_end_year)[0]

    if len(idx_tr) < 5 or len(idx_te) < 5:
        raise RuntimeError(
            f"Too few samples after split: train={len(idx_tr)}, test={len(idx_te)}"
        )

    train_mask_used = np.zeros(len(years_used), dtype=bool)
    train_mask_used[idx_tr] = True

    if DETREND_Y:
        Y_dt, (ay, by) = detrend_with_train_fit(
            years_used,
            Y,
            train_mask_used
        )
    else:
        Y_dt = Y.copy()
        ay, by = 0.0, 0.0

    if DETREND_X:
        X_dt = X.copy()
        for j in range(X_dt.shape[1]):
            X_dt[:, j], _ = detrend_with_train_fit(
                years_used,
                X[:, j],
                train_mask_used
            )
    else:
        X_dt = X.copy()

    return years_used, X_dt, Y_dt, idx_tr, idx_te, used, maxlag, (ay, by)


def format_axis(ax, years):
    ax.set_xlim(int(np.nanmin(years)), int(np.nanmax(years)) + 1)
    ax.xaxis.set_major_locator(MultipleLocator(20))
    ax.tick_params(axis="both", labelsize=TICK_FS)
    ax.grid(True, alpha=0.3)


# ============================================================
# MAIN RECONSTRUCTION FUNCTION
# ============================================================

def run_reconstruction_for_model(model):
    print("\n" + "=" * 80)
    print(f"MODEL: {model}")
    print("=" * 80)

    eof_dir = os.path.join(BASE_EOF_DIR, model, "latdepth_sections", "Train_period_85pct")
    best_csv = get_best_csv_path(model)
    outdir = get_model_outdir(model)

    print(f"BEST_CSV: {best_csv}")

    if not os.path.exists(best_csv):
        print(f"SKIPPING {model}: best_subsets_by_k.csv not found.")
        return None

    train_end_year = infer_train_end_year_from_any_eof(eof_dir)
    amoc_var_used = resolve_amoc_variable(TARGET, AMOC_VARIANT)

    print(f"TRAIN_END_YEAR: {train_end_year}")
    print(f"AMOC variable:   {amoc_var_used}")
    print(f"DETREND_Y:       {DETREND_Y}")
    print(f"DETREND_X:       {DETREND_X}")
    print(f"Ridge alpha:     {ALPHA_FIXED}")

    amoc = load_amoc(
        model=model,
        target=TARGET,
        mode=MODE,
        member_id=member_id,
        amoc_variant=AMOC_VARIANT
    )

    df_best = pd.read_csv(best_csv).sort_values("k")
    available_ks = sorted(df_best["k"].unique().tolist())
    k_max_use = min(K_MAX, int(max(available_ks)))

    print(f"Available k values: {available_ks}")
    print(f"Using k=1..{k_max_use}")

    all_feats_union = []

    for k in range(1, k_max_use + 1):
        row = df_best.loc[df_best["k"] == k]
        if len(row) != 1:
            continue
        feats = parse_features_string(row.iloc[0]["subset_features"])
        all_feats_union.extend(feats)

    needed_pairs = sorted(set((var, lon) for (var, lon, _, _) in all_feats_union))

    pc_dict = {
        (var, lon): load_pc_latdepth(
            model=model,
            var=var,
            lon_tag=lon,
            n_modes=N_MODES,
            mode=MODE,
            member_id=member_id
        )
        for (var, lon) in needed_pairs
    }

    common_years = amoc["year"].values.astype(int)

    for pc in pc_dict.values():
        common_years = np.intersect1d(
            common_years,
            pc["year"].values.astype(int)
        )

    years = np.asarray(common_years, int)
    years.sort()

    y_amoc = amoc.sel(year=years).values.astype(float)
    pc_np = {
        key: pc.sel(year=years).values.astype(float)
        for key, pc in pc_dict.items()
    }

    print(f"Common years: {years[0]}–{years[-1]} (n={len(years)})")

    k_rows = []
    pred_store = {}

    for k in range(1, k_max_use + 1):
        row = df_best.loc[df_best["k"] == k]

        if len(row) != 1:
            print(f"Skipping k={k}: not unique in best CSV.")
            continue

        feats_k = parse_features_string(row.iloc[0]["subset_features"])

        years_used, X_dt, Y_dt, idx_tr, idx_te, used, maxlag, (ay, by) = build_XY_for_feats(
            feats=feats_k,
            years_arr=years,
            y_amoc=y_amoc,
            pc_np=pc_np,
            train_end_year=train_end_year
        )

        model_pipe = make_pipeline(
            StandardScaler(),
            Ridge(alpha=ALPHA_FIXED)
        )

        Xtr = X_dt[idx_tr]
        Ytr = Y_dt[idx_tr]
        Xte = X_dt[idx_te]
        Yte = Y_dt[idx_te]

        m_tr = finite_rows_mask(Ytr, Xtr)
        m_te = finite_rows_mask(Yte, Xte)

        model_pipe.fit(Xtr[m_tr], Ytr[m_tr])

        pred_tr = model_pipe.predict(Xtr)
        pred_te = model_pipe.predict(Xte)

        r2_train = safe_r2(Ytr[m_tr], pred_tr[m_tr])
        r2_test = safe_r2(Yte[m_te], pred_te[m_te])
        corr_train = pearson_corr(Ytr[m_tr], pred_tr[m_tr])
        corr_test = pearson_corr(Yte[m_te], pred_te[m_te])

        y_pred_all = np.full(len(years), np.nan)
        y_pred_all[used[idx_tr]] = pred_tr
        y_pred_all[used[idx_te]] = pred_te

        if DETREND_Y:
            y_true_plot = y_amoc - (ay * years.astype(float) + by)
            ylab = "AMOC anomaly"
        else:
            y_true_plot = y_amoc
            ylab = "AMOC strength (Sv)"

        pred_store[k] = {
            "years": years.copy(),
            "y_true": y_true_plot.copy(),
            "y_pred": y_pred_all.copy(),
            "ylab": ylab,
        }

        feature_string = " | ".join(
            [f"{var} {lon} EOF{mode0 + 1} lag{lag}" for var, lon, mode0, lag in feats_k]
        )

        k_rows.append({
            "model": model,
            "k": k,
            "alpha": ALPHA_FIXED,
            "train_r2": r2_train,
            "test_r2": r2_test,
            "train_corr": corr_train,
            "test_corr": corr_test,
            "maxlag": maxlag,
            "years_used_start": int(years_used[0]),
            "years_used_end": int(years_used[-1]),
            "n_train": int(len(idx_tr)),
            "n_test": int(len(idx_te)),
            "features": feature_string,
        })

        print("\n" + "-" * 70)
        print(f"k = {k}")
        print(f"Train R² = {r2_train:.3f} | Test R² = {r2_test:.3f}")
        print(f"Train r  = {corr_train:.3f} | Test r  = {corr_test:.3f}")
        print(f"Max lag  = {maxlag}")
        print("Predictors:")
        for i, ft in enumerate(feats_k, start=1):
            print(f"  {i}. {human_readable_feature(ft)}")

    df_summary = pd.DataFrame(k_rows)
    summary_path = os.path.join(outdir, "reconstruction_summary_by_k.csv")
    df_summary.to_csv(summary_path, index=False)

    print("\nSaved summary:")
    print(summary_path)

    # ========================================================
    # Plot combined reconstruction figure
    # ========================================================
    ks_plot = sorted(pred_store.keys())

    if len(ks_plot) == 0:
        print(f"No predictions to plot for {model}.")
        return df_summary

    best_k = int(
        df_summary.loc[df_summary["test_r2"].idxmax(), "k"]
    )

    years0 = pred_store[ks_plot[0]]["years"]
    y_true0 = pred_store[ks_plot[0]]["y_true"]
    ylab0 = pred_store[ks_plot[0]]["ylab"]

    fig, ax = plt.subplots(figsize=FIGSIZE)

    ax.plot(
        years0,
        y_true0,
        color="black",
        linewidth=LW_AMOC,
        label=f"AMOC at {LATITUDE}",
        zorder=4
    )

    for k in ks_plot:
        label = f"k={k}"
        if k == best_k:
            label += " (best)"

        ax.plot(
            pred_store[k]["years"],
            pred_store[k]["y_pred"],
            color=COLOR_MAP.get(k, "grey"),
            linewidth=2.8 if k == best_k else LW_RECON,
            alpha=1.0 if k == best_k else 0.85,
            label=label,
            zorder=5 if k == best_k else 2
        )

    ax.axvline(
        train_end_year,
        linestyle="--",
        color="grey",
        linewidth=1.5,
        alpha=0.8,
        label="Train/test split"
    )

    ax.set_xlabel("Year", fontsize=LABEL_FS)
    ax.set_ylabel(ylab0, fontsize=LABEL_FS)
    ax.set_title(f"{model}: AMOC reconstruction at {LATITUDE}", fontsize=LABEL_FS)

    format_axis(ax, years0)

    ax.legend(
        loc="upper left",
        fontsize=LEGEND_FS,
        frameon=True,
        framealpha=0.95,
        facecolor="white",
        edgecolor="0.8"
    )

    fig.tight_layout()

    outbase = os.path.join(outdir, f"{model}_AMOC_reconstruction_k1to{k_max_use}")
    fig.savefig(outbase + ".png", dpi=300, bbox_inches="tight")
    fig.savefig(outbase + ".pdf", bbox_inches="tight")

    plt.show()

    print("\nSaved figure:")
    print(outbase + ".png")
    print(outbase + ".pdf")

    return df_summary


# ============================================================
# RUN ALL MODELS
# ============================================================

all_summaries = []

for model in MODELS:
    df_model = run_reconstruction_for_model(model)
    if df_model is not None:
        all_summaries.append(df_model)

if len(all_summaries) > 0:
    df_all = pd.concat(all_summaries, ignore_index=True)

    all_summary_path = os.path.join(
        OUT_BASE,
        f"all_models_reconstruction_summary_Detrended_{DETREND_Y}.csv"
    )

    df_all.to_csv(all_summary_path, index=False)

    print("\n" + "=" * 80)
    print("ALL MODEL SUMMARY")
    print("=" * 80)
    print(
        df_all[
            ["model", "k", "train_r2", "test_r2", "train_corr", "test_corr", "features"]
        ].to_string(index=False)
    )

    print("\nSaved combined summary:")
    print(all_summary_path)
else:
    print("No model reconstructions were produced.")

## EWS from selected features

In [ ]:
# ============================================================
# Early Warning Signals for selected reconstruction predictors
# EWS-only notebook script
# ============================================================

import os
import re
import glob
import itertools
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt

from matplotlib.ticker import MultipleLocator

os.environ["HDF5_USE_FILE_LOCKING"] = "FALSE"

# ============================================================
# USER SETTINGS
# ============================================================

MODELS = ["CESM2", "EC-Earth3", "IPSL-CM6A-LR", "MPI-ESM1-2-LR"]

TARGET = "AMOC_45N_ensmean"
LATITUDE = "45°N"

MODE = "ensmean"
member_id = None

AMOC_VARIANT = "normal"

# Must match reconstruction/subset-search setup
DETREND_Y = False
DETREND_X = False

SELECT_ON = "train"
MIN_LAG = 3
MAX_LAG = None

K_MAX = 3
N_MODES = 10
ALPHA_FIXED = 1.0

BASE_EOF_DIR = "/data/projects/nckf/frekle/EOF_results"
BASE_AMOC_DIR = "/data/users/frekle/AMOC_analysis"

OUT_BASE = "/data/users/frekle/Final_figures/Notebook_EWS_selected_predictors"
os.makedirs(OUT_BASE, exist_ok=True)

# EWS settings
AR1_WINDOW_YEARS = 30
AR1_MIN_VALID = 10

VAR_WINDOW_YEARS = 30
VAR_MIN_VALID = 10
VAR_DDOF = 1
STANDARDIZE_VARIANCE_INPUT = True

FIGSIZE = (12, 8.2)

LABEL_FS = 16
TICK_FS = 13
LEGEND_FS = 11
TITLE_FS = 16

LW_PRED = 2.2
LW_AMOC = 2.8


# ============================================================
# HELPERS
# ============================================================

def extract_years(time_coord):
    try:
        return xr.DataArray(time_coord).dt.year.values.astype(int)
    except Exception:
        return np.array([int(str(x)[:4]) for x in np.asarray(time_coord)], dtype=int)


def infer_train_end_year_from_any_eof(eof_dir, prefer=("so", "W30p0")):
    var, lon = prefer
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon}.nc")

    if not os.path.exists(f):
        hits = sorted(glob.glob(os.path.join(eof_dir, "EOF_latdepth_*_*.nc")))
        if not hits:
            raise FileNotFoundError(f"No EOF files found in {eof_dir}")
        f = hits[0]

    ds = xr.open_dataset(f)

    if "TRAIN_END" in ds.attrs:
        y = int(str(ds.attrs["TRAIN_END"])[:4])
        ds.close()
        return y

    if "train_mask" in ds:
        tm = ds["train_mask"].values.astype(bool)
        if tm.any():
            t_last = ds["time"].values[np.where(tm)[0][-1]]
            ds.close()
            return int(str(np.datetime64(t_last))[:4])

    ds.close()
    raise RuntimeError(f"Could not infer TRAIN_END year from {f}")


def infer_amoc_lat_from_target(target_name):
    if "26N" in target_name:
        return "26N"
    if "45N" in target_name:
        return "45N"
    raise ValueError(f"Could not infer latitude from TARGET={target_name}")


def resolve_amoc_variable(target, amoc_variant):
    if amoc_variant == "normal":
        return target
    if amoc_variant == "smooth":
        lat = infer_amoc_lat_from_target(target)
        return f"AMOC_{lat}_smooth"
    raise ValueError("AMOC_VARIANT must be 'normal' or 'smooth'")


def load_amoc(model, target, mode="ensmean", member_id=None, amoc_variant="normal"):
    amoc_file = os.path.join(BASE_AMOC_DIR, f"AMOC_{model}.nc")
    varname = resolve_amoc_variable(target, amoc_variant)

    ds = xr.open_dataset(amoc_file)

    if varname not in ds.data_vars:
        available = list(ds.data_vars)
        ds.close()
        raise KeyError(f"{varname} not found in {amoc_file}. Available: {available}")

    y = ds[varname].squeeze()

    if "year" in y.dims:
        yy = y
        if "time" in yy.coords:
            yy = yy.drop_vars("time")
    else:
        years_tmp = extract_years(y["time"])
        yy = (
            y.assign_coords(year=("time", years_tmp))
             .swap_dims({"time": "year"})
             .drop_vars("time")
        )

    yy = yy.astype(float)

    if "member" in yy.dims:
        if mode == "ensmean":
            yy = yy.mean("member")
        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when MODE='member'")

            if "member" in yy.coords and member_id in yy["member"].values:
                yy = yy.sel(member=member_id)
            else:
                yy = yy.isel(member=int(member_id))
        else:
            ds.close()
            raise ValueError("MODE must be 'ensmean' or 'member'")

    ds.close()
    return yy.squeeze()


def load_pc_latdepth(model, var, lon_tag, n_modes, mode="ensmean", member_id=None):
    eof_dir = os.path.join(BASE_EOF_DIR, model, "latdepth_sections", "Train_period_85pct")
    f = os.path.join(eof_dir, f"EOF_latdepth_{var}_{lon_tag}.nc")

    if not os.path.exists(f):
        raise FileNotFoundError(f"Missing EOF file: {f}")

    ds = xr.open_dataset(f)

    if "PC" not in ds:
        ds.close()
        raise KeyError(f"'PC' not found in {f}")

    pc = ds["PC"].isel(mode=slice(0, int(n_modes)))

    if "member" in pc.dims:
        if mode == "ensmean":
            pc = pc.mean("member")
        elif mode == "member":
            if member_id is None:
                ds.close()
                raise ValueError("member_id must be provided when MODE='member'")

            if "member" in pc.coords and member_id in pc["member"].values:
                pc = pc.sel(member=member_id)
            else:
                pc = pc.isel(member=int(member_id))
        else:
            ds.close()
            raise ValueError("MODE must be 'ensmean' or 'member'")

    pc = pc.transpose("time", "mode")
    years_tmp = extract_years(pc["time"])

    pc = (
        pc.assign_coords(year=("time", years_tmp))
          .swap_dims({"time": "year"})
          .drop_vars("time")
          .astype(float)
    )

    ds.close()
    return pc


def parse_features_string(s):
    feats = []

    for part in str(s).split("|"):
        part = part.strip()
        m = re.search(r"(\w+)\s+(W\d+p\d+)\s+EOF(\d+)\s+lag(\d+)", part)

        if not m:
            continue

        var = m.group(1)
        lon = m.group(2)
        mode0 = int(m.group(3)) - 1
        lag = int(m.group(4))

        feats.append((var, lon, mode0, lag))

    out = []
    seen = set()

    for ft in feats:
        key = (ft[0], ft[1], ft[2], ft[3])
        if key not in seen:
            out.append(ft)
            seen.add(key)

    return out


def unique_modes_from_features(feats):
    out = []
    seen = set()

    for var, lon, mode0, lag in feats:
        key = (var, lon, mode0)
        if key not in seen:
            out.append((var, lon, mode0))
            seen.add(key)

    return out


def fit_linear_trend(train_years, train_series):
    x = np.asarray(train_years, float)
    y = np.asarray(train_series, float)

    m = np.isfinite(x) & np.isfinite(y)
    x = x[m]
    y = y[m]

    if len(x) < 2:
        return 0.0, float(np.nanmean(y))

    A = np.vstack([x, np.ones_like(x)]).T
    a, b = np.linalg.lstsq(A, y, rcond=None)[0]

    return float(a), float(b)


def detrend_with_train_fit(all_years, all_series, train_mask):
    all_years = np.asarray(all_years, float)
    all_series = np.asarray(all_series, float)

    a, b = fit_linear_trend(
        all_years[train_mask],
        all_series[train_mask]
    )

    return all_series - (a * all_years + b), (a, b)


def standardize_with_train_fit(all_series, train_mask):
    x = np.asarray(all_series, float)

    train_x = x[train_mask]
    train_x = train_x[np.isfinite(train_x)]

    if len(train_x) < 2:
        mu = float(np.nanmean(x))
        sigma = float(np.nanstd(x))
    else:
        mu = float(np.nanmean(train_x))
        sigma = float(np.nanstd(train_x, ddof=1))

    if not np.isfinite(sigma) or sigma == 0:
        sigma = 1.0

    return (x - mu) / sigma, (mu, sigma)


def pearson_corr(a, b):
    a = np.asarray(a, float)
    b = np.asarray(b, float)

    m = np.isfinite(a) & np.isfinite(b)

    if m.sum() < 3:
        return np.nan

    a = a[m]
    b = b[m]

    if a.std() == 0 or b.std() == 0:
        return np.nan

    return float(np.corrcoef(a, b)[0, 1])


def rolling_ar1(x, years_arr, window=30, min_valid=10):
    x = np.asarray(x, float)
    years_arr = np.asarray(years_arr, int)

    n = len(x)
    if n < window:
        window = max(5, n // 2)

    ar1_vals = np.full(n, np.nan, float)
    half = window // 2

    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)

        seg = x[lo:hi]
        seg = seg[np.isfinite(seg)]

        if len(seg) < max(min_valid, 3):
            continue

        a = seg[1:]
        b = seg[:-1]

        if len(a) < 2:
            continue

        ar1_vals[i] = pearson_corr(a, b)

    return years_arr, ar1_vals


def rolling_variance(x, years_arr, window=30, min_valid=10, ddof=1):
    x = np.asarray(x, float)
    years_arr = np.asarray(years_arr, int)

    n = len(x)
    if n < window:
        window = max(5, n // 2)

    var_vals = np.full(n, np.nan, float)
    half = window // 2

    for i in range(n):
        lo = max(0, i - half)
        hi = min(n, i + half + 1)

        seg = x[lo:hi]
        seg = seg[np.isfinite(seg)]

        if len(seg) < max(min_valid, 2):
            continue

        if len(seg) <= ddof:
            continue

        var_vals[i] = float(np.var(seg, ddof=ddof))

    return years_arr, var_vals


def pretty_var_name(v):
    if str(v).lower() == "thetao":
        return "Temperature"
    if str(v).lower() == "so":
        return "Salinity"
    return str(v)


def pretty_lon_name(lon_tag):
    m = re.match(r"W(\d+)p(\d+)", str(lon_tag))
    if m:
        whole = int(m.group(1))
        dec = int(m.group(2))
        if dec == 0:
            return f"{whole}°W"
        return f"{whole}.{dec}°W"
    return str(lon_tag)


def short_feature_label(var, lon, mode0):
    return f"{pretty_var_name(var)}, {pretty_lon_name(lon)}, EOF{mode0 + 1}"


def get_best_csv_path(model):
    maxlag_tag = MAX_LAG if MAX_LAG is not None else "None"

    return (
        f"/data/users/frekle/Final_figures/{model}/{TARGET}/Feature_selection/"
        f"Detrended_{DETREND_Y}/Selected_on_{SELECT_ON}/amoc_variant_{AMOC_VARIANT}/"
        f"lag_policy_{MIN_LAG}minlag_{maxlag_tag}max/best_subsets_by_k.csv"
    )


def get_model_outdir(model):
    outdir = os.path.join(
        OUT_BASE,
        model,
        TARGET,
        f"Detrended_{DETREND_Y}",
        f"amoc_variant_{AMOC_VARIANT}",
        f"minlag_{MIN_LAG}"
    )

    os.makedirs(outdir, exist_ok=True)
    return outdir


def build_common_years_and_pc_np(model, selected_modes, amoc):
    needed_pairs = sorted(set((var, lon) for var, lon, _ in selected_modes))

    pc_dict = {
        (var, lon): load_pc_latdepth(
            model=model,
            var=var,
            lon_tag=lon,
            n_modes=N_MODES,
            mode=MODE,
            member_id=member_id
        )
        for var, lon in needed_pairs
    }

    common_years = amoc["year"].values.astype(int)

    for pc in pc_dict.values():
        common_years = np.intersect1d(
            common_years,
            pc["year"].values.astype(int)
        )

    years = np.asarray(common_years, int)
    years.sort()

    y_amoc = amoc.sel(year=years).values.astype(float)

    pc_np = {
        key: pc.sel(year=years).values.astype(float)
        for key, pc in pc_dict.items()
    }

    return years, y_amoc, pc_np


def build_ews_series_for_feature(var, lon, mode0, years, pc_np, train_mask):
    pc_series = pc_np[(var, lon)][:, int(mode0)].copy()

    if DETREND_X:
        pc_series, _ = detrend_with_train_fit(
            years,
            pc_series,
            train_mask
        )

    ar1_years, ar1_vals = rolling_ar1(
        pc_series,
        years,
        window=AR1_WINDOW_YEARS,
        min_valid=AR1_MIN_VALID
    )

    var_input = pc_series.copy()

    if STANDARDIZE_VARIANCE_INPUT:
        var_input, _ = standardize_with_train_fit(
            var_input,
            train_mask
        )

    var_years, var_vals = rolling_variance(
        var_input,
        years,
        window=VAR_WINDOW_YEARS,
        min_valid=VAR_MIN_VALID,
        ddof=VAR_DDOF
    )

    return {
        "pc": pc_series,
        "ar1_years": ar1_years,
        "ar1": ar1_vals,
        "var_years": var_years,
        "variance": var_vals,
    }


def build_ews_series_for_amoc(years, y_amoc, train_mask):
    y_series = y_amoc.copy()

    if DETREND_Y:
        y_series, _ = detrend_with_train_fit(
            years,
            y_series,
            train_mask
        )

    ar1_years, ar1_vals = rolling_ar1(
        y_series,
        years,
        window=AR1_WINDOW_YEARS,
        min_valid=AR1_MIN_VALID
    )

    var_input = y_series.copy()

    if STANDARDIZE_VARIANCE_INPUT:
        var_input, _ = standardize_with_train_fit(
            var_input,
            train_mask
        )

    var_years, var_vals = rolling_variance(
        var_input,
        years,
        window=VAR_WINDOW_YEARS,
        min_valid=VAR_MIN_VALID,
        ddof=VAR_DDOF
    )

    return {
        "series": y_series,
        "ar1_years": ar1_years,
        "ar1": ar1_vals,
        "var_years": var_years,
        "variance": var_vals,
    }


def summarize_ews_values(label, ews):
    ar1 = np.asarray(ews["ar1"], float)
    var = np.asarray(ews["variance"], float)

    out = {
        "label": label,
        "ar1_mean": np.nanmean(ar1),
        "ar1_min": np.nanmin(ar1),
        "ar1_max": np.nanmax(ar1),
        "var_mean": np.nanmean(var),
        "var_min": np.nanmin(var),
        "var_max": np.nanmax(var),
        "valid_ar1_points": int(np.isfinite(ar1).sum()),
        "valid_var_points": int(np.isfinite(var).sum()),
    }

    return out


# ============================================================
# MAIN FUNCTION
# ============================================================

def run_ews_for_model(model):
    print("\n" + "=" * 80)
    print(f"EWS ANALYSIS: {model}")
    print("=" * 80)

    eof_dir = os.path.join(BASE_EOF_DIR, model, "latdepth_sections", "Train_period_85pct")
    best_csv = get_best_csv_path(model)
    outdir = get_model_outdir(model)

    print(f"BEST_CSV: {best_csv}")

    if not os.path.exists(best_csv):
        print(f"SKIPPING {model}: best_subsets_by_k.csv not found.")
        return None

    train_end_year = infer_train_end_year_from_any_eof(eof_dir)
    amoc_var_used = resolve_amoc_variable(TARGET, AMOC_VARIANT)

    print(f"TRAIN_END_YEAR: {train_end_year}")
    print(f"AMOC variable:   {amoc_var_used}")
    print(f"DETREND_Y:       {DETREND_Y}")
    print(f"DETREND_X:       {DETREND_X}")
    print(f"AR1 window:      {AR1_WINDOW_YEARS} years")
    print(f"Variance window: {VAR_WINDOW_YEARS} years")
    print(f"Variance input standardized: {STANDARDIZE_VARIANCE_INPUT}")

    df_best = pd.read_csv(best_csv).sort_values("k")
    available_ks = sorted(df_best["k"].unique().tolist())
    k_max_use = min(K_MAX, int(max(available_ks)))

    print(f"Available k values: {available_ks}")
    print(f"Using selected features from k=1..{k_max_use}")

    all_selected_features = []

    for k in range(1, k_max_use + 1):
        row = df_best.loc[df_best["k"] == k]

        if len(row) != 1:
            print(f"Skipping k={k}: not unique in best CSV.")
            continue

        feats_k = parse_features_string(row.iloc[0]["subset_features"])
        all_selected_features.extend(feats_k)

        print(f"\nk={k} selected predictors:")
        for ft in feats_k:
            var, lon, mode0, lag = ft
            print(f"  - {short_feature_label(var, lon, mode0)}, lag {lag}")

    selected_modes = unique_modes_from_features(all_selected_features)

    print("\nUnique predictor modes used for EWS:")
    for var, lon, mode0 in selected_modes:
        print(f"  - {short_feature_label(var, lon, mode0)}")

    if len(selected_modes) == 0:
        print(f"No selected predictor modes found for {model}.")
        return None

    amoc = load_amoc(
        model=model,
        target=TARGET,
        mode=MODE,
        member_id=member_id,
        amoc_variant=AMOC_VARIANT
    )

    years, y_amoc, pc_np = build_common_years_and_pc_np(
        model=model,
        selected_modes=selected_modes,
        amoc=amoc
    )

    train_mask = years <= train_end_year

    print(f"\nCommon years: {years[0]}–{years[-1]} (n={len(years)})")
    print(f"Train years:  {years[train_mask][0]}–{years[train_mask][-1]}")
    print(f"Test years:   {years[~train_mask][0]}–{years[~train_mask][-1]}")

    # AMOC EWS
    amoc_ews = build_ews_series_for_amoc(
        years=years,
        y_amoc=y_amoc,
        train_mask=train_mask
    )

    summary_rows = []

    summary_rows.append({
        "model": model,
        "feature": f"AMOC at {LATITUDE}",
        **summarize_ews_values(f"AMOC at {LATITUDE}", amoc_ews)
    })

    feature_ews_store = {}

    for var, lon, mode0 in selected_modes:
        label = short_feature_label(var, lon, mode0)

        ews = build_ews_series_for_feature(
            var=var,
            lon=lon,
            mode0=mode0,
            years=years,
            pc_np=pc_np,
            train_mask=train_mask
        )

        feature_ews_store[(var, lon, mode0)] = ews

        summary_rows.append({
            "model": model,
            "feature": label,
            **summarize_ews_values(label, ews)
        })

    df_summary = pd.DataFrame(summary_rows)
    summary_path = os.path.join(outdir, "ews_summary_selected_predictors.csv")
    df_summary.to_csv(summary_path, index=False)

    print("\nEWS summary:")
    print(
        df_summary[
            ["feature", "ar1_mean", "ar1_min", "ar1_max",
             "var_mean", "var_min", "var_max",
             "valid_ar1_points", "valid_var_points"]
        ].to_string(index=False)
    )

    print("\nSaved EWS summary:")
    print(summary_path)

    # ========================================================
    # Plot AR1 + variance figure
    # ========================================================
    fig, axes = plt.subplots(
        nrows=2,
        ncols=1,
        figsize=FIGSIZE,
        sharex=True,
        gridspec_kw={"hspace": 0.08}
    )

    ax_ar1, ax_var = axes

    color_cycle = itertools.cycle(plt.cm.tab10.colors)

    for var, lon, mode0 in selected_modes:
        label = short_feature_label(var, lon, mode0)
        ews = feature_ews_store[(var, lon, mode0)]
        color = next(color_cycle)

        ax_ar1.plot(
            ews["ar1_years"],
            ews["ar1"],
            linewidth=LW_PRED,
            color=color,
            label=label
        )

        ax_var.plot(
            ews["var_years"],
            ews["variance"],
            linewidth=LW_PRED,
            color=color,
            label=label
        )

    ax_ar1.plot(
        amoc_ews["ar1_years"],
        amoc_ews["ar1"],
        linewidth=LW_AMOC,
        linestyle="--",
        color="black",
        label=f"AMOC at {LATITUDE}"
    )

    ax_var.plot(
        amoc_ews["var_years"],
        amoc_ews["variance"],
        linewidth=LW_AMOC,
        linestyle="--",
        color="black",
        label=f"AMOC at {LATITUDE}"
    )

    for ax in axes:
        ax.axvline(
            train_end_year,
            linestyle="--",
            color="grey",
            linewidth=1.4,
            alpha=0.8
        )

        ax.grid(True, alpha=0.3)
        ax.tick_params(axis="both", labelsize=TICK_FS)
        ax.xaxis.set_major_locator(MultipleLocator(20))
        ax.set_xlim(int(years.min()), int(years.max()) + 1)

    ax_ar1.set_ylabel("Lag-1 autocorrelation", fontsize=LABEL_FS)

    if STANDARDIZE_VARIANCE_INPUT:
        ax_var.set_ylabel("Rolling variance\nstandardized", fontsize=LABEL_FS)
    else:
        ax_var.set_ylabel("Rolling variance", fontsize=LABEL_FS)

    ax_var.set_xlabel("Year", fontsize=LABEL_FS)

    ax_ar1.set_title(
        f"{model}: EWS for selected reconstruction predictors",
        fontsize=TITLE_FS
    )

    handles, labels = ax_ar1.get_legend_handles_labels()

    unique = {}
    for h, lab in zip(handles, labels):
        if lab not in unique:
            unique[lab] = h

    fig.legend(
        list(unique.values()),
        list(unique.keys()),
        loc="lower center",
        ncol=min(3, max(1, len(unique))),
        frameon=True,
        framealpha=0.95,
        facecolor="white",
        edgecolor="0.8",
        fontsize=LEGEND_FS,
        bbox_to_anchor=(0.5, -0.01)
    )

    fig.subplots_adjust(
        left=0.12,
        right=0.98,
        top=0.93,
        bottom=0.20,
        hspace=0.08
    )

    outbase = os.path.join(
        outdir,
        f"{model}_EWS_selected_predictors_AR1_variance"
    )

    fig.savefig(outbase + ".png", dpi=300, bbox_inches="tight")
    fig.savefig(outbase + ".pdf", bbox_inches="tight")

    plt.show()

    print("\nSaved EWS figure:")
    print(outbase + ".png")
    print(outbase + ".pdf")

    return df_summary


# ============================================================
# RUN ALL MODELS
# ============================================================

all_ews_summaries = []

for model in MODELS:
    df_model = run_ews_for_model(model)

    if df_model is not None:
        all_ews_summaries.append(df_model)

if len(all_ews_summaries) > 0:
    df_all_ews = pd.concat(all_ews_summaries, ignore_index=True)

    all_summary_path = os.path.join(
        OUT_BASE,
        f"all_models_EWS_summary_Detrended_{DETREND_Y}.csv"
    )

    df_all_ews.to_csv(all_summary_path, index=False)

    print("\n" + "=" * 80)
    print("ALL MODEL EWS SUMMARY")
    print("=" * 80)

    print(
        df_all_ews[
            ["model", "feature", "ar1_mean", "ar1_min", "ar1_max",
             "var_mean", "var_min", "var_max"]
        ].to_string(index=False)
    )

    print("\nSaved combined EWS summary:")
    print(all_summary_path)

else:
    print("No EWS analyses were produced.")